In [2]:
## Setup

import sqlite3
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

def get_connection(city):
    return sqlite3.connect(f'../data/processed/{city}/{city}.db')

def query_db(conn, query):
    return pd.read_sql_query(query, conn)

# Check what columns exist in dim_listing
def check_columns(conn, table):
    cursor = conn.cursor()
    cursor.execute(f"PRAGMA table_info({table})")
    return [col[1] for col in cursor.fetchall()]

# Load data
nyc_conn = get_connection('nyc')

# Check dim_listing columns
dim_cols = check_columns(nyc_conn, 'dim_listing')
print(f"\ndim_listing columns: {dim_cols[:15]}...")

# Build query dynamically based on available columns
select_cols = []

# Map desired columns to actual column names
col_mapping = {
    'listing_id': 'listing_id',  # This is the actual column name in dim_listing
    'name': 'name',
    'description': 'description',
    'room_type': 'room_type',
    'property_type': 'property_type',
    'neighbourhood': 'neighbourhood',
    'accommodates': 'accommodates',
    'bedrooms': 'bedrooms',
    'beds': 'beds',
    'bathrooms': 'bathrooms',
    'amenities': 'amenities',
    'price': 'base_price',  # In dim_listing it's base_price
    'rating': 'rating',  # This might not exist in dim_listing
    'number_of_reviews': 'number_of_reviews'  # This might not exist in dim_listing
}

# Build SELECT clause
select_parts = []
for alias, col in col_mapping.items():
    if col in dim_cols:
        select_parts.append(f"{col} AS {alias}")
    else:
        select_parts.append(f"NULL AS {alias}")

select_clause = ",\n        ".join(select_parts)

query = f"""
SELECT 
    {select_clause}
FROM dim_listing l
WHERE base_price IS NOT NULL
  AND base_price < 1000
LIMIT 1000
"""

print(f"\nQuery:")
print(query[:200] + "...")

# Execute query
listings_df = query_db(nyc_conn, query)
nyc_conn.close()

print(f"\nLoaded {len(listings_df)} listings")
print(f"\nColumns: {listings_df.columns.tolist()}")
print(f"\nSample data:")
print(listings_df.head())


dim_listing columns: ['listing_key', 'listing_id', 'name', 'neighbourhood', 'room_type', 'property_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'base_price', 'host_id', 'host_name', 'host_is_superhost', 'host_tenure_years']...

Query:

SELECT 
    listing_id AS listing_id,
        name AS name,
        description AS description,
        room_type AS room_type,
        property_type AS property_type,
        neighbourhood AS neighb...

Loaded 1000 listings

Columns: ['listing_id', 'name', 'description', 'room_type', 'property_type', 'neighbourhood', 'accommodates', 'bedrooms', 'beds', 'bathrooms', 'amenities', 'price', 'rating', 'number_of_reviews']

Sample data:
   listing_id                                               name  \
0        6848                   Only 2 stops to Manhattan studio   
1        6872  Uptown Sanctuary w/ Private Bath (Month to Month)   
2        6990                            UES Beautiful Blue Room   
3        7097      Perfect for Your Parents,

In [3]:
## Data Cleaning

# Fill missing values
listings_df['room_type'] = listings_df['room_type'].fillna('Unknown')
listings_df['neighbourhood'] = listings_df['neighbourhood'].fillna('Unknown')
listings_df['property_type'] = listings_df['property_type'].fillna('Unknown')
listings_df['description'] = listings_df['description'].fillna('')
listings_df['name'] = listings_df['name'].fillna('Listing')
listings_df['amenities'] = listings_df['amenities'].fillna('')

# Convert numeric columns
numeric_cols = ['accommodates', 'bedrooms', 'beds', 'bathrooms', 'price']
for col in numeric_cols:
    if col in listings_df.columns:
        listings_df[col] = pd.to_numeric(listings_df[col], errors='coerce').fillna(0)

print(f"Data shape after cleaning: {listings_df.shape}")


Data shape after cleaning: (1000, 14)


In [4]:
## Feature Engineering

def create_content_string(row):
    """Create a string representation of listing features."""
    features = []
    
    # Basic features
    features.append(f"room_type_{row.get('room_type', 'unknown')}")
    features.append(f"property_type_{row.get('property_type', 'unknown')}")
    features.append(f"neighbourhood_{row.get('neighbourhood', 'unknown')}")
    
    # Numeric features as strings
    for col in ['accommodates', 'bedrooms', 'beds', 'bathrooms']:
        val = row.get(col, 0)
        if val > 0:
            features.append(f"{col}_{int(val)}")
    
    # Price range
    price = row.get('price', 0)
    if price > 0:
        if price < 100:
            features.append("price_budget")
        elif price < 200:
            features.append("price_mid")
        else:
            features.append("price_premium")
    
    # Name and description
    name = str(row.get('name', ''))
    features.extend(name.split()[:5])
    
    desc = str(row.get('description', ''))
    features.extend(desc.split()[:10])
    
    # Amenities (if available)
    amenities = str(row.get('amenities', ''))
    if amenities and amenities != 'nan':
        try:
            import ast
            # Try to parse as list
            if isinstance(amenities, str) and amenities.startswith('['):
                amenity_list = ast.literal_eval(amenities)
                features.extend([f"amenity_{a.replace(' ', '_')[:20]}" for a in amenity_list[:5]])
        except:
            pass
    
    return ' '.join([str(f).lower() for f in features if f and str(f) != 'nan'])

listings_df['content'] = listings_df.apply(create_content_string, axis=1)

print("\nContent features created")
print(f"Sample: {listings_df['content'].iloc[0][:300]}...")


Content features created
Sample: room_type_entire home/apartment property_type_entire rental unit neighbourhood_unknown accommodates_3 bedrooms_2 price_mid only 2 stops to manhattan comfortable studio apartment with super comfortable king size bed and amenity_microwave amenity_wifi amenity_hangers amenity_essentials amenity_stove...


In [5]:
## TF-IDF Vectorization

# Vectorize content
vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
content_matrix = vectorizer.fit_transform(listings_df['content'])

print(f"Content matrix shape: {content_matrix.shape}")

# Compute similarity matrix
similarity_matrix = cosine_similarity(content_matrix)

print(f"Similarity matrix shape: {similarity_matrix.shape}")

Content matrix shape: (1000, 500)
Similarity matrix shape: (1000, 1000)


In [ ]:
## TF-IDF Vectorization

# Vectorize content
vectorizer = TfidfVectorizer(max_features=500, stop_words='english')
content_matrix = vectorizer.fit_transform(listings_df['content'])

print(f"Content matrix shape: {content_matrix.shape}")

# Compute similarity matrix
similarity_matrix = cosine_similarity(content_matrix)

print(f"Similarity matrix shape: {similarity_matrix.shape}")

In [6]:
## Recommendation Function

def recommend_listings(listing_id, n_recommendations=5):
    """Recommend similar listings."""
    # Find listing in DataFrame
    mask = listings_df['listing_id'] == listing_id
    if not mask.any():
        return f"Listing {listing_id} not found"
    
    idx = listings_df[mask].index[0]
    
    # Get similarity scores
    similarity_scores = list(enumerate(similarity_matrix[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)
    
    # Get top recommendations (excluding self)
    top_indices = [i[0] for i in similarity_scores[1:n_recommendations+1]]
    top_scores = [i[1] for i in similarity_scores[1:n_recommendations+1]]
    
    # Get recommendations
    recommendations = listings_df.iloc[top_indices][
        ['listing_id', 'name', 'room_type', 'neighbourhood', 'price']
    ].copy()
    recommendations['similarity_score'] = top_scores
    
    return recommendations

# Test recommendation
if len(listings_df) > 0:
    test_listing = listings_df['listing_id'].iloc[0]
    print(f"\nRecommendations for listing {test_listing}:")
    print(f"Original: {listings_df.iloc[0]['name']}")
    recs = recommend_listings(test_listing, 5)
    if isinstance(recs, pd.DataFrame):
        print(recs[['listing_id', 'name', 'price', 'similarity_score']].to_string(index=False))
else:
    print("No listings available for recommendation")


Recommendations for listing 6848:
Original: Only 2 stops to Manhattan studio
 listing_id                                name  price  similarity_score
    4760107        Tall Ceilings, Large Windows 119.51          0.472983
    1220548       Cozy 2 BR apartment in Queens  71.99          0.444853
     144087               LUXURY OF THE HORIZON 255.83          0.428172
    3760029 Charming studio for rent in Astoria 144.05          0.415483
    1984486   Comfortable Brown Stone apartment 129.08          0.412295


In [7]:
## Recommendation System Evaluation

def evaluate_recommendations(n_samples=10):
    """Evaluate recommendation quality."""
    if len(listings_df) < 10:
        print("Not enough listings for evaluation")
        return 0
    
    scores = []
    
    for i in range(min(n_samples, len(listings_df))):
        listing_id = listings_df['listing_id'].iloc[i]
        recs = recommend_listings(listing_id, 3)
        
        if isinstance(recs, pd.DataFrame) and len(recs) > 0:
            # Check if recommendations have similar price range
            original_price = listings_df.iloc[i]['price']
            avg_rec_price = recs['price'].mean()
            
            if original_price > 0:
                price_diff = abs(original_price - avg_rec_price) / original_price
                price_score = max(0, 1 - price_diff)
                scores.append(price_score)
    
    if scores:
        avg_score = np.mean(scores)
        print(f"\nAverage recommendation quality score: {avg_score:.3f}")
        return avg_score
    else:
        print("No evaluation data available")
        return 0

avg_score = evaluate_recommendations(20)

print("\n💡 BUSINESS INTERPRETATION:")
if avg_score > 0.7:
    print("✅ Recommendations are high quality (similar price range)")
elif avg_score > 0.5:
    print("⚠️ Recommendations are moderately accurate")
else:
    print("❌ Recommendations need improvement with more features")


Average recommendation quality score: 0.573

💡 BUSINESS INTERPRETATION:
⚠️ Recommendations are moderately accurate


In [9]:
## Save Recommendation System

import pickle
import os
from pathlib import Path

# Create models directory if it doesn't exist
models_dir = Path('../models')
models_dir.mkdir(parents=True, exist_ok=True)

print(f"📁 Models directory: {models_dir.absolute()}")

# Save vectorizer
vectorizer_path = models_dir / 'vectorizer.pkl'
with open(vectorizer_path, 'wb') as f:
    pickle.dump(vectorizer, f)
print(f"✅ Vectorizer saved to: {vectorizer_path}")

# Save similarity matrix
similarity_path = models_dir / 'similarity_matrix.pkl'
with open(similarity_path, 'wb') as f:
    pickle.dump(similarity_matrix, f)
print(f"✅ Similarity matrix saved to: {similarity_path}")

# Save listings data
listings_path = models_dir / 'listings_data.parquet'
listings_df.to_parquet(listings_path)
print(f"✅ Listings data saved to: {listings_path}")

# Also save the recommendation function for later use
import inspect

# Save a small sample for testing
sample_listings = listings_df.head(100).copy()
sample_listings.to_parquet(models_dir / 'sample_listings.parquet')
print(f"✅ Sample listings saved to: {models_dir / 'sample_listings.parquet'}")

print("\n" + "="*60)
print("✅ Recommendation system saved successfully!")
print("="*60)
print(f"\nFiles saved in: {models_dir.absolute()}")
print("  - vectorizer.pkl")
print("  - similarity_matrix.pkl")
print("  - listings_data.parquet")
print("  - sample_listings.parquet")

📁 Models directory: c:\Users\ANUJI THRIMANNA\Downloads\airbnb-assessment\notebooks\..\models
✅ Vectorizer saved to: ..\models\vectorizer.pkl
✅ Similarity matrix saved to: ..\models\similarity_matrix.pkl
✅ Listings data saved to: ..\models\listings_data.parquet
✅ Sample listings saved to: ..\models\sample_listings.parquet

✅ Recommendation system saved successfully!

Files saved in: c:\Users\ANUJI THRIMANNA\Downloads\airbnb-assessment\notebooks\..\models
  - vectorizer.pkl
  - similarity_matrix.pkl
  - listings_data.parquet
  - sample_listings.parquet
